In [39]:
from pyspark.sql import functions as F

BRONZE = "business_bronze"
SILVER = "business_silver"

df_business = spark.table(BRONZE)


StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 41, Finished, Available, Finished, False)

In [33]:
base_cols = [
    "business_id","name","address","city","state","postal_code",
    "latitude","longitude","stars","review_count","is_open","categories",
    "_ingest_ts","_source_file","_batch_id"
]

df_business_silver = df_business.select(*base_cols)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 35, Finished, Available, Finished, False)

In [34]:
df_business_silver = (df_business_silver
    .withColumn("latitude", F.col("latitude").cast("double"))
    .withColumn("longitude", F.col("longitude").cast("double"))
    .withColumn("stars", F.col("stars").cast("double"))
    .withColumn("review_count", F.col("review_count").cast("int"))
    .withColumn("is_open", F.col("is_open").cast("int"))

    .dropDuplicates(["business_id"])

    .filter(F.col("business_id").isNotNull())
    .filter(F.col("stars").isNotNull())
    .filter((F.col("stars")>=0)&(F.col("stars")<=5))
    .filter(F.col("review_count").isNotNull())
    .filter(F.col("review_count")>=0)
    .filter(F.col("is_open").isNotNull())
    .filter(F.col("is_open").isin(0, 1))
)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 36, Finished, Available, Finished, False)

In [35]:
(df_business_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 37, Finished, Available, Finished, False)

In [36]:
df_business_silver.printSchema()

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 38, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- stars: double (nullable = true)
 |-- review_count: integer (nullable = true)
 |-- is_open: integer (nullable = true)
 |-- categories: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [37]:
print("business_silver rows: ", df_business_silver.count())

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 39, Finished, Available, Finished, False)

business_silver rows:  150346


In [38]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, e740649c-51a5-487a-b7c4-e2e231ff6721, 40, Finished, Available, Finished, False)

ExitValue: SUCCESS